Mapeamento Regional e Temporal: Identificar quais regiões e estados do Brasil apresentam as maiores taxas de crimes contra mulheres e a população LGBTQIA+, bem como analisar se há uma tendência de aumento ou redução desses índices ao longo dos últimos anos.


Avaliação de Políticas Públicas: Avaliar, através de dados, se a implementação de leis de proteção específicas resultou em impactos reais na diminuição das taxas de criminalidade nas regiões analisadas.

Análise de Tipificação: Investigar a qualidade dos dados oficiais, buscando indícios de falhas de tipificação (como registrar crimes de ódio como crimes comuns).

Implicação Prática: Fornecer um panorama analítico que possa auxiliar gestores de segurança e organizações da sociedade civil na alocação de recursos (como delegacias especializadas) e no aprimoramento dos sistemas de registro de boletins de ocorrência.

# Datas:

**Início da criminalização de crimes contra LGBTQ**:

Desde junho de 2019, o STF, na ADO nº 26, criminalizou a LGBTfobia no Brasil, enquadrando atos de ódio e discriminação contra orientação sexual ou identidade de gênero na Lei de Racismo (Lei 7.716/1989). A conduta é crime inafiançável e imprescritível, com penas de 1 a 3 anos de prisão, podendo ser maior em casos de injúria racial. 
Homicídio Qualificado/Hediondo: A discriminação que motiva um homicídio (LGBTIcídio) torna o crime hediondo, com pena maior (12 a 30 anos).

**Lei do Feminicídio**:

A Lei do Feminicídio (Lei nº 13.104/2015), atualizada pela Lei nº 14.994/2024, torna o assassinato de mulheres por razões de gênero um crime autônomo e hediondo, com penas severas de 20 a 40 anos de reclusão. Considera-se feminicídio quando envolve violência doméstica/familiar ou menosprezo/discriminação à condição de mulher, com agravantes para crimes cometidos na gravidez, contra menores de 14 ou maiores de 60 anos, ou na presença de familiares.

# Perguntas

Quais regiões e estados do Brasil concentram as maiores taxas de violência contra mulheres e pessoas LGBTQIA+, e existe padrão de sazonalidade ou escalonada ao longo dos anos?

Há uma correlação observável entre a implementação de marcos legais de proteção (como a Lei do Feminicídio ou a criminalização da homofobia pelo STF) e a diminuição das ocorrências em estados específicos?

A proporção de feminicídios em relação ao total de homicídios de mulheres indica uma subnotificação ou falha na tipificação do crime em determinadas bases de dados estaduais?

Como a ausência de campos específicos para orientação sexual e identidade de gênero nos registros oficiais impacta a dimensão dos dados quando comparados aos levantamentos de ONGs e observatórios independentes?

In [1]:
from pathlib import Path

import requests

import pandas as pd

project_root = Path.cwd()
while not (project_root / "pyproject.toml").exists() and project_root != project_root.parent:
    project_root = project_root.parent


In [2]:
def download_xlsx(url: str, year: int, download_dir: str | Path) -> Path:
    download_dir = Path(download_dir)
    download_dir.mkdir(parents=True, exist_ok=True)

    download_url = url.format(y=year)
    file_path = download_dir / f"pubseg-{year}.xlsx"
    response = requests.get(download_url, timeout=60)
    response.raise_for_status()
    file_path.write_bytes(response.content)

    return file_path

In [3]:
def table_ajust_pubseg(db: pd.DataFrame):

    year_series = pd.to_datetime(db["data_referencia"], errors="coerce").dt.year

    if year_series.isna().any():
        raise ValueError("The 'data_referencia' column must contain valid dates to generate the id_tabela.")
    
    row_id = pd.Series(range(1, len(db) + 1), index=db.index, dtype="int64")
    db["id_tabela"] = year_series.astype("Int64").astype("string") + row_id.astype("string")
    ordered_columns = ["id_tabela", *[column for column in db.columns if column != "id_tabela"]]
    db = db[ordered_columns]

    db["feminino"] = (
        pd.to_numeric(db["feminino"], errors="coerce")  # garante numérico
        .fillna(0)                                     # NaN → 0
        .astype(int)
    )
    db["masculino"] = (
        pd.to_numeric(db["masculino"], errors="coerce")  # garante numérico
        .fillna(0)                                     # NaN → 0
        .astype(int)
    )
    db["nao_informado"] = (
        pd.to_numeric(db["nao_informado"], errors="coerce")  # garante numérico
        .fillna(0)                                     # NaN → 0
        .astype(int)
    )
    db["total_vitima"] = (
        pd.to_numeric(db["total_vitima"], errors="coerce")
        .fillna(0)
        .astype(int)
    )

    return db

In [4]:
def tabel_ajust_disk(db):
    db.columns = (
        db.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
    )

    db["data_de_cadastro"] = pd.to_datetime(
        db["data_de_cadastro"], format="%d/%m/%Y %H:%M", errors="coerce"
    )
    db["sl_quantidade_vitimas"] = pd.to_numeric(
        db["sl_quantidade_vitimas"], errors="coerce"
    ).astype("Int64")
    db["id_tabela"] = db["data_de_cadastro"].dt.strftime("%Y%m%d%H%M%S")

In [6]:
year = ["segundo-semestre-de-2025", "primeiro-semestre-de-2025"]
url = "https://www.gov.br/mdh/pt-br/acesso-a-informacao/dados-abertos/disque100/{y}"
download_path = project_root / "data" / "csv_dowloads"

schema_name = "public"
table_name = "raw_disk100"

path100_1=download_xlsx(url, year[0], download_path)
path100_2=download_xlsx(url, year[1], download_path)

year = 2025
url = "https://www.gov.br/mj/pt-br/assuntos/sua-seguranca/seguranca-publica/estatistica/download/dnsp-base-de-dados/bancovde-{y}.xlsx/@@download/file"
download_path = project_root / "data" / "xlsx_dowloads"

path_pubseg = download_xlsx(url, year, download_path)



HTTPError: 404 Client Error: Not Found for url: https://www.gov.br/mdh/pt-br/acesso-a-informacao/dados-abertos/disque100/segundo-semestre-de-2025

In [ ]:
disk100_1 = pd.read_csv(path100_1, sep=";", encoding="latin1")
disk100_2 = pd.read_csv(path100_2, sep=";", encoding="latin1")
pubseg = pd.read_excel(path_pubseg)

In [ ]:
disk100_1 = tabel_ajust_disk(disk100_1)
disk100_2 = tabel_ajust_disk(disk100_2)
pubseg = table_ajust_pubseg(pubseg)